In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

%matplotlib inline


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/Abhijeet/Documents/coding-repos/nn-pytorch-mastery/.venv/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/Abhijeet/Documents/coding-repos/nn-pytorch-mastery/.venv/lib/python3.11/site-packages/traitlets/config/application.py", line 1080, in launch_instance
    app.start()
  File "/Users/Abhijeet/Documents/coding-repos/nn-pytorch-mastery/

In [2]:
words = open('names.txt', 'r').read().splitlines()
words[:8], len(words)

(['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia'],
 32033)

In [3]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [4]:
print(chars)
print(stoi)
print(itos)

['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}
{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [5]:
X = []
Y = []

block_size = 3

for w in words:
    context = [0] * block_size

    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

In [6]:
X = torch.tensor(X)
Y = torch.tensor(Y)

In [7]:
# print(X)
# print(Y)
print(X.shape)
print(Y.shape)

torch.Size([228146, 3])
torch.Size([228146])


In [8]:
C = torch.randn(27,10)
C.shape
print(X[0])
print(X[0].shape)

emb = C[X[0]]

print(emb.shape)

tensor([0, 0, 0])
torch.Size([3])
torch.Size([3, 10])


In [9]:
W1 = torch.randn(30, 200)
b1 = torch.randn(200)

W2 = torch.randn((200, 27))
b2 = torch.randn(27)

C = torch.randn(27,10)

parameters = [C, W1, b1, W2, b2]

for p in parameters:
    p.requires_grad = True

In [10]:
emb = C[X[2]]
emb_flat = emb.view(-1, 30)

z = emb_flat @ W1
print("z:", z.shape)
z = z + b1
print("z after bias:", z.shape)
h = torch.tanh(z)
print("h:", h.shape)
logits = h @ W2 + b2
logits.shape, logits

z: torch.Size([1, 200])
z after bias: torch.Size([1, 200])
h: torch.Size([1, 200])


(torch.Size([1, 27]),
 tensor([[ -2.5627, -21.9082, -12.7251,  20.9079,   4.4272,  -1.0337,  18.3384,
          -24.9084,  17.7198,  14.2237, -20.2662,   9.4808,   8.0706, -16.0158,
            7.0620, -22.0893,   3.1021,  -7.5317,   0.8608,  -0.1742, -11.3091,
           14.3207,  11.8261,   5.6868,   3.1980, -14.7648,  -5.5508]],
        grad_fn=<AddBackward0>))

In [11]:
counts = logits.exp()
counts.shape, counts

(torch.Size([1, 27]),
 tensor([[7.7095e-02, 3.0577e-10, 2.9756e-06, 1.2027e+09, 8.3693e+01, 3.5570e-01,
          9.2106e+07, 1.5220e-11, 4.9614e+07, 1.5042e+06, 1.5794e-09, 1.3106e+04,
          3.1990e+03, 1.1077e-07, 1.1668e+03, 2.5511e-10, 2.2245e+01, 5.3584e-04,
          2.3651e+00, 8.4010e-01, 1.2261e-05, 1.6573e+06, 1.3678e+05, 2.9494e+02,
          2.4483e+01, 3.8702e-07, 3.8845e-03]], grad_fn=<ExpBackward0>))

In [12]:
prob = counts / counts.sum(1, keepdims=True)  # why do we need, keepdims here
prob.shape, prob

(torch.Size([1, 27]),
 tensor([[5.7202e-11, 2.2687e-19, 2.2078e-15, 8.9239e-01, 6.2097e-08, 2.6391e-10,
          6.8339e-02, 1.1293e-20, 3.6811e-02, 1.1160e-03, 1.1718e-18, 9.7243e-06,
          2.3735e-06, 8.2191e-17, 8.6569e-07, 1.8928e-19, 1.6505e-08, 3.9758e-13,
          1.7548e-09, 6.2333e-10, 9.0972e-15, 1.2296e-03, 1.0148e-04, 2.1883e-07,
          1.8166e-08, 2.8715e-16, 2.8822e-12]], grad_fn=<DivBackward0>))

In [13]:

target = Y[2]

loss_manual = -prob[0, target].log()

loss_pytorch = F.cross_entropy(
    logits,
    target.unsqueeze(0)
)

print("manual:", loss_manual.item())
print("PyTorch:", loss_pytorch.item())

manual: 37.0374870300293
PyTorch: 37.03749084472656


In [14]:

loss = F.cross_entropy(logits, target.unsqueeze(0))
loss

tensor(37.0375, grad_fn=<NllLossBackward0>)

In [15]:
loss.backward()

In [16]:
print(W2.grad.shape)
print(W1.grad.shape)
print(b2.grad.shape)
print(b1.grad.shape)
print(C.grad.shape)

torch.Size([200, 27])
torch.Size([30, 200])
torch.Size([27])
torch.Size([200])
torch.Size([27, 10])


In [17]:
print(W2.grad[0, 0])
print(W2[0, 0])

tensor(-5.7200e-11)
tensor(-0.0039, grad_fn=<SelectBackward0>)


In [18]:
idx = (0, 0)
old = W2[idx].item()
old

-0.003944675903767347

In [19]:
eps = 1

W2.data[idx] = old + eps

# recompute forward pass
emb = C[X[2]]
h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
logits_plus = h @ W2 + b2
loss_plus = F.cross_entropy(logits_plus, target.unsqueeze(0))

W2.data[idx] = old

In [20]:
numerical_grad = (loss_plus.item() - loss.item())/eps

print("autograd:", W2.grad[idx].item())
print("numerical:", numerical_grad)

autograd: -5.719973106277365e-11
numerical: 0.0


In [21]:
n1 = int(0.8 * len(X))
n2 = int(0.9 * len(X))

Xtr, Ytr = X[:n1], Y[:n1]
Xdev, Ydev = X[n1:n2], Y[n1:n2]
Xte, Yte = X[n2:], Y[n2:]

Xtr.shape, Ytr.shape

(torch.Size([182516, 3]), torch.Size([182516]))

In [22]:
ix = torch.randint(0, Xtr.shape[0], (32,))

In [23]:
ix

tensor([ 80700, 141207,  41411,  94994,   2319,  49512, 131740, 141514, 158503,
         23657, 105992,  58054,  33523, 148246, 120527, 109789, 133766, 181949,
         75207, 109876, 171590,  36297, 182186,    202, 132946,  66361, 171967,
        161283,  82120,   2977, 129277,  17921])

In [24]:
Xtr[ix].shape , Ytr[ix].shape

(torch.Size([32, 3]), torch.Size([32]))

In [25]:
emb = C[Xtr[ix]]
emb.view(-1, 30).shape

torch.Size([32, 30])

In [26]:
## Starting form SCRATCH

g = torch.Generator().manual_seed(2147483647)

C = torch.randn((27, 10), generator=g)
W1 = torch.randn((30, 200), generator=g)
b1 = torch.randn(200, generator=g)

W2 = torch.randn((200, 27), generator=g)
b2 = torch.randn(27, generator=g)

parameters = [C, W1, b1, W2, b2]

for p in parameters:
    p.requires_grad = True

In [27]:
for i in range(200000):

    # minibatch
    ix = torch.randint(0, Xtr.shape[0], (32,))

    # forward pass
    emb = C[Xtr[ix]]
    h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
    logits = h @ W2 + b2

    # loss
    loss = F.cross_entropy(logits, Ytr[ix])

    # zero gradients
    for p in parameters:
        p.grad = None

    # backward pass
    loss.backward()

    # learning rate schedule
    lr = 0.1 if i < 100000 else 0.01

    # update
    for p in parameters:
        p.data += -lr * p.grad

    if i % 10000 == 0:
        print(i, loss.item())

0 24.618680953979492
10000 2.9581680297851562
20000 2.112701654434204
30000 2.2446470260620117
40000 2.1035361289978027
50000 2.2388980388641357
60000 2.2817654609680176
70000 2.4276087284088135
80000 2.4742319583892822
90000 2.3994557857513428
100000 2.4692788124084473
110000 2.5569231510162354
120000 2.1838157176971436
130000 1.8901253938674927
140000 2.130458116531372
150000 1.5792816877365112
160000 2.1481406688690186
170000 1.8614922761917114
180000 2.277236223220825
190000 2.3051929473876953


In [28]:
loss_tr = F.cross_entropy(
    torch.tanh(C[Xtr].view(-1, 30) @ W1 + b1) @ W2 + b2,
    Ytr
)

loss_dev = F.cross_entropy(
    torch.tanh(C[Xdev].view(-1, 30) @ W1 + b1) @ W2 + b2,
    Ydev
)

loss_te = F.cross_entropy(
    torch.tanh(C[Xte].view(-1, 30) @ W1 + b1) @ W2 + b2,
    Yte
)

print("train:", loss_tr.item())
print("dev:", loss_dev.item())
print("test:", loss_te.item())

train: 2.0640666484832764
dev: 2.395246744155884
test: 2.4332470893859863


In [29]:
context = torch.tensor([[0, 0, 0]])

with torch.no_grad():
    emb = C[context]
    h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
    logits = h @ W2 + b2
    probs = F.softmax(logits, dim=1)
    ix = torch.multinomial(probs, num_samples=1)

print("logits:", logits.shape)
print("probs:", probs.shape)
print("sampled index:", ix)
print("sampled character:", itos[ix.item()])

context = torch.tensor([[0, 0, ix]])

with torch.no_grad():
    emb = C[context]
    h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
    logits = h @ W2 + b2
    probs = F.softmax(logits, dim=1)
    ix = torch.multinomial(probs, num_samples=1).item()

print(ix, itos[ix])

logits: torch.Size([1, 27])
probs: torch.Size([1, 27])
sampled index: tensor([[1]])
sampled character: a
13 m


In [30]:
context = [0, 0, 0]
out = []

while True:

    x = torch.tensor([context])

    with torch.no_grad():
        emb = C[x]
        h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)

        ix = torch.multinomial(
            probs,
            num_samples=1
        ).item()

    # stop condition
    if ix == 0:
        break

    # save character
    out.append(itos[ix])

    # slide context window
    context = context[1:] + [ix]

print(''.join(out))

wesume
